In [1]:
import numpy as np
import pandas as pd
import json
import datetime
import pickle

In [2]:
# load in psiturk data
rm1df = pd.read_json('../../data/db/exported/room1-2.11.19.json', convert_dates=['beginhit','endhit'])
rm2df = pd.read_json('../../data/db/exported/room2-2.11.19.json', convert_dates=['beginhit','endhit'])

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1]
rm2df = rm2df[rm2df.status != 1]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)
rm2df = rm2df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)

# format datastring as dict 
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2

# remove test runs for each room
rm1df = rm1df.loc[1:]
rm2df = rm2df.loc[1:]

# concatenate dataframes, order by start time
expdf = pd.concat([rm1df, rm2df], ignore_index=True).sort_values('beginhit').reset_index(drop=True)

# drop subjects 
expdf = expdf.loc[expdf.index != 7].reset_index(drop=True) # experiment error
expdf = expdf.loc[expdf.index != 56].reset_index(drop=True) # no-show for part 2
expdf = expdf.loc[expdf.index != 62].reset_index(drop=True) # no-show for part 2

In [42]:
expdf.loc[expdf.uniqueid == 'debugokLIG:debugalG88']

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom
7,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3,1


In [3]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../data/google-form-data/Pre-experiment Questionnaire.csv', parse_dates=[0])
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
postqdf = pd.read_csv('../../data/google-form-data/Post-experiment questionnaire.csv', parse_dates=[0])
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)

In [4]:
# convert from datetime to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
# add 3 hours to convert from UTC to ET
# subtract 1 hours to EST times to account for datetime handling of DST
newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
for ix, val in enumerate(newpretimestamp):
    newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
    if ix <= 92:
        newpretimestamp[ix] += 3.6e6

preqdf['preqtime'] = newpretimestamp

newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
for ix, val in enumerate(newposttimestamp):
    newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
    if ix <= 92:
        newposttimestamp[ix] += 3.6e6

postqdf['postqtime'] = newposttimestamp

In [5]:
# remove dropped subjects from google form
dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-1011318-A-05','MD-020119-B-01',
           'MD-101218-B-04']

for dropid in dropids:
    preqdf = preqdf[preqdf['Subject ID'] != dropid]
    postqdf = postqdf[postqdf['Subject ID'] != dropid]
        
preqdf.reset_index(drop=True, inplace=True)
postqdf.reset_index(drop=True, inplace=True)

In [6]:
# fix mistaken day/repeat ID assignments

preqdf.at[76,'Subject ID'] = 'MD-102018-A-03'
postqdf.at[76,'Subject ID'] = 'MD-102018-A-03'

preqdf.at[77,'Subject ID'] = 'MD-102218-A-06'
postqdf.at[77,'Subject ID'] = 'MD-102218-A-06'

preqdf.at[85,'Subject ID'] = 'MD-102218-B-06'
postqdf.at[85,'Subject ID'] = 'MD-102218-B-06'

preqdf.at[89,'Subject ID'] = 'MD-013119-A-02'
postqdf.at[90,'Subject ID'] = 'MD-013119-A-02'

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [7]:
# add empty columns from pre/postquestionnaires to expdf
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

# separate session 1 and 2 postquestionnaires
ses1postqdf = postqdf.drop_duplicates('Subject ID')
tempdf = postqdf.copy(deep=True)
rowsin1 = [ix for ix, row in ses1postqdf.iterrows()]

for ix, row in tempdf.iterrows():
    if ix in rowsin1:
        tempdf.drop(ix, inplace=True)
ses2postqdf = tempdf

ses1postqdf.reset_index(drop=True, inplace=True)
ses2postqdf.reset_index(drop=True, inplace=True)

In [8]:
# add prequestionnaire responses to corresponding subject
for ix, row in expdf.iterrows():
    for col in preqdf.columns:
        expdf.at[ix,col] = preqdf.at[ix,col]

# add session 1 postquestionnaire to first occurence of each subject
for ix, row in expdf.drop_duplicates('Subject ID').iterrows():
    expdf.loc[ix,'postqtime':] = (ses1postqdf.loc[ses1postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

# add session 2 postquestionnaire to rest
for ix, row in expdf.iterrows():
    if np.isnan(row['postqtime']):
        expdf.loc[ix,'postqtime':] = (ses2postqdf.loc[ses2postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

In [9]:
# drop subject -- self-reported difficuly with task due to sudden onset migraine and lack of sleep
expdf = expdf.loc[expdf['Subject ID'] != 'MD-102218-A-04'].reset_index(drop=True)

In [10]:
# drop subjects who didn't sufficiently complete task
expdf = expdf.loc[expdf['Subject ID'] != 'MD-101318-A-01'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-102218-B-06'].reset_index(drop=True)
expdf = expdf.loc[expdf['Subject ID'] != 'MD-020119-B-03'].reset_index(drop=True)

In [13]:
id_maps = {}

for ix, row in expdf.iterrows():
    if row['Subject ID'] in id_maps:
        id_maps[row['Subject ID']]['session 2'] = row['uniqueid']#.replace('/',':')
    else:
        id_maps[row['Subject ID']] = {'session 1' : row['uniqueid'], 'session 2' : None}

In [12]:
with open('../../data/pickles/expdf.p', 'wb') as file:
    pickle.dump(expdf, file)

In [17]:
with open('../../data/pickles/id_maps.p', 'wb') as file:
    pickle.dump(id_maps, file)

In [14]:
# # for checking correct mapping between experiment data and google forms
# for ix, row in expdf.iterrows():
#     for time in [row['datastring']['data'][0]['dateTime'], row['preqtime']]:
#         print(datetime.datetime.fromtimestamp(time/1e3))
#     print(ix)
#     print('___________')

In [22]:
import os

In [38]:
audioids = os.listdir('../../data/audio/room1') + os.listdir('../../data/audio/room2')
dictids = []
for sid, data in id_maps.items():
    for ses, tid in data.items():
        if tid is not None:
            dictids.append(tid)

['debugokLIG:debugalG88',
 'debugcY6kj:debug80qcA',
 'debugKTkCo:debugh4yBd',
 '.DS_Store',
 'debugbICiy:debugKisj0',
 'debugLakdV:debugAP6bj',
 'debug6j0oH:debuge4lNg',
 'debugQzo2F:debugV7e7L',
 'debugxQ7lq:debugS1UNb',
 'debugLF5EW:debugTFi6y',
 'debugMDnZj:debugPx77x',
 'debugGVTD3:debugfpzCT',
 'debugzcmcY:debuguPcuX',
 'debugT1dDc:debugG8Zuf',
 'debugBbcu2:debugep6I7',
 'debugHNBk3:debugUUMPo',
 '.DS_Store',
 'debug2LQvu:debugk4zJe',
 'debugV24oG:debug1XLSo',
 'debugFIZ10:debugxB6Uy',
 'debug41SJm:debugJW4AU',
 'debugAN3bL:debugJClT8',
 'debugmRIPW:debugYLZyM',
 'debugb8Mbo:debugWMuG3',
 'debugWyz8d:debugWt3x1',
 'debugM7hoQ:debugTzCVu']

In [14]:
id_maps

{'MD-013119-A-01': {'session 1': 'debughKp9W:debug1QFCO',
  'session 2': 'debugSeRNG:debug5pTo9'},
 'MD-013119-A-02': {'session 1': 'debug5ytRS:debug22nvn',
  'session 2': 'debug3Z74G:debug6CQ6D'},
 'MD-013119-B-01': {'session 1': 'debuguhHBa:debug1HOCZ',
  'session 2': 'debug0GSk9:debugttOEY'},
 'MD-020119-A-02': {'session 1': 'debugfGFHT:debugLCOgS',
  'session 2': 'debug9ngWi:debugpMjGl'},
 'MD-020119-A-03': {'session 1': 'debugRQQWb:debugwDi2H',
  'session 2': 'debugWRfIG:debugJWnAi'},
 'MD-020119-A-04': {'session 1': 'debugo1MpX:debugfpXYH',
  'session 2': 'debugeyj0U:debugsaXvf'},
 'MD-020119-B-02': {'session 1': 'debugAU1cu:debugzdGmZ',
  'session 2': 'debug27k0b:debugOGsyi'},
 'MD-020719-B-01': {'session 1': 'debugwHDQh:debugDtNml', 'session 2': None},
 'MD-020819-B-01': {'session 1': 'debug3RFCM:debuggg1Gs', 'session 2': None},
 'MD-101218-A-01': {'session 1': 'debugIEH2T:debugDLVLJ',
  'session 2': 'debug2Ea7T:debugosNZ7'},
 'MD-101218-A-02': {'session 1': 'debugYQfMB:debugxg